In [1]:
import os
import pandas as pd
from pathlib import Path
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (NTXpred2)

This notebook curates the **NTXpred2** dataset by aggregating multiple Excel files into a single standardized table. \
The raw inputs contain peptide sequences and associated neurotoxicity labels, which are merged, quality-controlled, \
and deduplicated before exporting a clean neurotoxic dataset and its corresponding metadata.

- **Toxic effect / endpoint:** neurotoxic
- **Source:** NTXpred2
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads multiple Excel files** from the NTXpred2 source directory:
  - skips non-data header rows,
  - enforces a consistent column schema (`sequence`, `label`).
- **Concatenates all sources** into a unified DataFrame.
- **Checks duplicated sequences**:
  - identical sequences with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends dataset-level statistics.
- **Exports curated outputs**:
  - `processed_neurotoxic_dataset.csv`,
  - `metadata.json`.

In [2]:
name_source = "NTXpred2"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
dfs = []

for file in (Path(PATH_INPUT) / name_source).glob("*"):
    df = pd.read_excel(file, skiprows=1, names=["sequence", "label"])
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
df.shape

(4848, 2)

- Checking duplicates

In [4]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [5]:
df_full.shape

(3300, 2)

In [6]:
df_errors.shape

(0, 1)

- Working with metada

In [7]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [8]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'GNU general public license',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 2, 21, 0, 0),
 'download date': Timestamp('2025-03-01 00:00:00'),
 'file format': 'csv',
 'peptide property': 'neurotoxic, toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from Swiss-Prot',
 'repository or server': 'https://webs.iiitd.edu.in/raghava/ntxpred2/download.html',
 'publication': 'https://www.biorxiv.org/content/10.1101/2025.03.01.640936v1.full',
 'number_of_raw_sequences': 4848,
 'number_of_sequences_retained': 3300,
 'number_of_positive_sequences': 1649,
 'number_of_negative_sequences': 1651,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [9]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [10]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_neurotoxic_dataset.csv", index=False)